In [1]:
%load_ext autoreload
%autoreload 2

import pickle
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Iterable

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import navis
from tqdm import tqdm

from src.neuron import POSTSYNAPSE_TYPE, PRESYNAPSE_TYPE, Neuron
from src.graph import EigendecomposedMatrixStrategy

# Zed incorrecly shows tables when same index is skipped
pd.set_option("display.multi_sparse", False)

In [2]:
cell_types = pd.read_csv("data/consolidated_cell_types.csv.gz")
classification = pd.read_csv("data/classification.csv.gz")
connectivity_tags = pd.read_csv("data/connectivity_tags.csv.gz")
nt_predictions = pd.read_csv("data/nt_predictions.csv.gz")

from functools import reduce

neurons = reduce(
    lambda left_df, right_df: pd.merge(left_df, right_df, on="root_id", how="inner"),
    [cell_types, classification, connectivity_tags, nt_predictions],
).set_index("root_id").sort_index()
neurons

,primary_type,additional_type(s),flow,super_class,class,sub_class,hemilineage,side,nerve,connectivity_tag,group,nt_type,nt_type_score,da_avg,ser_avg,gaba_avg,glut_avg,ach_avg,oct_avg
root_id,,,,,,,,,,,,,,,,,,,
720575940596125868,T5c,NaN,intrinsic,optic,optic_lobe_intrinsic,t5_neuron,NaN,right,NaN,"reciprocal,feedforward_loop_participant",LO.LOP,ACH,0.57,0.03,0.00,0.05,0.28,0.57,0.07
720575940597856265,Tm16,NaN,intrinsic,optic,optic_lobe_intrinsic,transmedullary,NaN,right,NaN,"rich_club,reciprocal,feedforward_loop_particip...",ME,ACH,0.85,0.01,0.00,0.03,0.04,0.85,0.07
720575940597944841,Tm7,NaN,intrinsic,optic,optic_lobe_intrinsic,transmedullary,NaN,right,NaN,"rich_club,reciprocal,feedforward_loop_particip...",ME.LO,ACH,0.82,0.02,0.00,0.01,0.05,0.82,0.09
720575940598267657,TmY15,NaN,intrinsic,optic,optic_lobe_intrinsic,transmedullary_y,NaN,right,NaN,"rich_club,reciprocal,feedforward_loop_particip...",ME,GABA,0.74,0.01,0.01,0.74,0.19,0.03,0.02
720575940599333574,Tm1,NaN,intrinsic,optic,optic_lobe_intrinsic,transmedullary,NaN,right,NaN,"rich_club,reciprocal,feedforward_loop_particip...",ME,ACH,0.61,0.02,0.00,0.13,0.13,0.61,0.11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
720575940661335681,CB0180,NaN,intrinsic,central,NaN,NaN,MX12__prim,left,NaN,"rich_club,reciprocal,feedforward_loop_particip...",GNG,GABA,0.71,0.00,0.00,0.71,0.17,0.11,0.00
720575940661336193,TmY3,NaN,intrinsic,optic,optic_lobe_intrinsic,transmedullary_y,NaN,left,NaN,"rich_club,reciprocal,feedforward_loop_particip...",ME.LO,ACH,0.62,0.02,0.00,0.18,0.16,0.62,0.03
720575940661337217,KCab-p,NaN,intrinsic,central,Kenyon_Cell,NaN,MBp4,left,NaN,"rich_club,reciprocal,feedforward_loop_particip...",SCL.MB_ML,ACH,1.00,0.00,0.00,0.00,0.00,1.00,0.00


In [3]:
synapse_coordinates = pd.read_csv(
    "data/synapse_coordinates.csv.gz",
    dtype={
        "pre_root_id": "Int64",
        "post_root_id": "Int64",
    }
)
synapse_coordinates[["pre_root_id", "post_root_id"]] = synapse_coordinates[["pre_root_id", "post_root_id"]].ffill().astype("int64")
synapse_coordinates = synapse_coordinates.set_index(["pre_root_id", "post_root_id"]).sort_index()

synapse_coordinates

,,x,y,z
pre_root_id,post_root_id,,,
720575940596125868,720575940608552405,705988,275676,221800
720575940596125868,720575940608552405,706908,275084,222160
720575940596125868,720575940608552405,707848,275456,222100
720575940596125868,720575940608552405,708050,275176,222200
720575940596125868,720575940608552405,711344,267664,223360
...,...,...,...,...
720575940661337217,720575940652390134,436160,163856,81040
720575940661337217,720575940652390134,436370,165082,78360
720575940661337217,720575940652390134,438644,167636,77860


In [4]:
UNKNOWN_NODE_ID = -1

connectors = synapse_coordinates.copy().assign(
    connector_id=range(len(synapse_coordinates)),
    node_id=UNKNOWN_NODE_ID,
    type=None,
)

connector_index_for = {
    PRESYNAPSE_TYPE: "pre_root_id",
    POSTSYNAPSE_TYPE: "post_root_id",
}


def get_connectors(root_id: int, type) -> pd.DataFrame:    
    try:
        neuron_connectors = connectors.xs(root_id, level=connector_index_for[type])
    except KeyError:
        return connectors.iloc[0:0]
        
    neuron_connectors["type"] = type
    return neuron_connectors


connectors

,,x,y,z,connector_id,node_id,type
pre_root_id,post_root_id,,,,,,
720575940596125868,720575940608552405,705988,275676,221800,0,-1,None
720575940596125868,720575940608552405,706908,275084,222160,1,-1,None
720575940596125868,720575940608552405,707848,275456,222100,2,-1,None
720575940596125868,720575940608552405,708050,275176,222200,3,-1,None
720575940596125868,720575940608552405,711344,267664,223360,4,-1,None
...,...,...,...,...,...,...,...
720575940661337217,720575940652390134,436160,163856,81040,34156315,-1,None
720575940661337217,720575940652390134,436370,165082,78360,34156316,-1,None
720575940661337217,720575940652390134,438644,167636,77860,34156317,-1,None


In [7]:
def load_skeleton(root_id: int) -> Neuron:
    skeleton = Neuron.from_swc(root_id)
    
    skeleton.connectors = pd.concat([
        get_connectors(root_id, type=PRESYNAPSE_TYPE),
        get_connectors(root_id, type=POSTSYNAPSE_TYPE),
    ]).reset_index(drop=True)

    skeleton.nt_type = neurons.loc[root_id]["nt_type"]

    return skeleton


skeletons = []
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(load_skeleton, root_id) for root_id in neurons.head(50000).index]
    
    for future in tqdm(as_completed(futures), total=len(neurons)):
        skeletons.append(future.result())

print(f"Loaded {len(skeletons)} neuron skeletons")

 37%|█████████████████████████████████████████▊                                                                      | 50000/133846 [03:58<06:40, 209.36it/s]

Loaded 50000 neuron skeletons


In [20]:
strategy = EigendecomposedMatrixStrategy(skeletons)
strategy.fire(*[skeletons[i].root_id for i in range(100)])

In [ ]:
strategy.step()

50000it [00:28, 1757.90it/s]


Calculate effects


In [ ]:
strategy.plot(activity_threshold=0.01)